# Notebook - ATAC-seq and scATAC-seq (in-class)

We thank the ENCODE consortium as well as Satija and Stuart team for their software and package tutorials, from which much of the material below is adapted. We thank the Epigenomics Workshop 2024 for posting educational materials online, which were also adapted in creating this notebook. 

## [[Important!!]] Instruction for running this notebook on Cocalc:
One of the files types that we need to access which contains data for us (`.h5`) requires that we install a library that R can use to read them.

First, let's try to see if the required library is available and can be loaded in your notebook. Execute the following command: 

In [ ]:
library("hdf5r")

We also want to make sure we're using the most updated versions of each package. Run the following command to check the version of `Signac` that you have installed:

In [ ]:
packageVersion("Signac")

If `hdf5r` loads without error and your `Signac` version >= 1.15.0, then you are good to go! You can skip the section below and head straight to "Loading Packages" below.

If however either condition above is not met, we'll need to install some libraries and packages to get this configured and working on your own environment. 

---

**Public Service Announcement**: Note that below, we will use the `sudo` preamble here. `sudo` is a *very special command* in UNIX that you only are able to use here on CoCalc because you have administrator permissions for your own project space! Basically, it allows you to perform work as an administrator (allowing you to access directories a user normally could not, for safety purposes). In most cloud environments, for systems protections this permission will **not** be available to you and you will have to request installations from systems administrators.

To install the required LINUX libraries, we wil use `apt`, aka APT or the Advanced Package Tool. This is a free-software command-line and library package management system used on Debian, Ubuntu, Linux Mint, and other Unix-like Linux distributions.

---

With that PSA out of the way, follow the steps below: 

1. Head to the "View" menu, scroll down and select "Terminal". This will split your screen and give you a terminal prompt.
2. In that terminal, execute the following UNIX command (note "$>" is the UNIX prompt, not something you have to enter):

`$> sudo apt update && sudo apt install libhdf5-dev`

3. When prompted, enter "Y" (or yes) to proceed. The install should take less than 30 seconds.
4. Next, we will need to install a package in R. After the install is complete, run the following UNIX command:

`$> sudo R`

This will bring up an R console on the command line.

5. within that console, run the following R commands:

`install.packages("hdf5r")`

`install.packages("Signac")`

6. Verify that `hdf5r` has installed correctly and loads without complaint by entering the following command:

`library(hdf5r)`

7. Verify that an updated version of `Signac` has installed correctly (this should return `TRUE`):

`packageVersion("Signac") > "1.15.0"`

7. You can exit the R console with `q()` and close the terminal split screen.

8. As a last step, let's restart the R kernel in this notebook as well, to make sure it can see all of the items you have installed. Go to the "Kernel" menu item and select "Restart Kernel". This should take a few seconds. At this point, you should be good to proceed with notebook!

## Loading packages

Returning to this notebook, let's now load libraries that we have installed to check that the setup is working.

First, let's load our libraries in two groups:

In [ ]:
# run this
library(Seurat)
library(Signac)
library(tidyverse)

In [ ]:
# now run this
library(biovizBase)
library(EnsDb.Hsapiens.v86)
library(patchwork)
library("presto")

In [ ]:
# Now run this to set up for annotations
library(AnnotationHub)

In [ ]:
# Now run this library to set up enrichment analyses
library(clusterProfiler)
library(org.Hs.eg.db)
library(enrichplot)

## Objectives of this notebook

The objective of this notebook is to get students familiar with data analysis on bulk and single-cell ATAC-seq data. This notebook will guide students through the steps of quality control, clustering, exploratory data analysis, and differential accessibility analysis. By the end of this process, we aim to uncover insights into the chromatin accessibility landscape across different conditions or cell types, and be able to process both public and primary datasets.

## Set up pipelines for bulk ATAC-seq data analysis

The ENCODE consortium provides a very useful tool for analyzing the bulk ATAC-seq data. Please take a look at their workflow, and answer Q1. https://www.encodeproject.org/pipelines/ENCPL787FUN/ (we recommend opening the website using Safari or another browser, as you may encounter display issues with Chrome).

**Q1.** What are the input and output of the pipeline? 

To run the ENCODE pipeline, a `.json` input file is required. All the experimental summary information, `.fastq` file location, and R1 and R2 read file locations should be provided in this file in order for the pipeline to recognize them. Here, we will not actually run the whole pipeline, but we will edit the following `.json` file to mimic what we'll need to do when running the pipeline. 

Instructions for editing the input `.json` file can be found here: https://github.com/ENCODE-DCC/atac-seq-pipeline/blob/master/docs/input_short.md.

**Q2.** Please read the above instructions and make corrections to the `.json` file in the text block below, with the information provided here:

1. The data are collected from mouse liver
    - Hint: which genome should you be referencing? Are your title and description accurate?)
2. You don't know the adapter, so you want the algorithm to automatically detect the adapter 
    - Hint: which field below seems to set if adapters are automatically detected?
3. Double check the input `.fastq` files
    - Hint: Does each set of paired reads have matching lane files in the correct order?
    - Hint: Does each replicate have the correct number of corresponding adapters per lane?

In [ ]:
{
    "atac.title" : "Example (paired end)",
    "atac.description" : "This is a template input JSON for paired end sample.",

    "atac.pipeline_type" : "atac",
    "atac.align_only" : false,
    "atac.true_rep_only" : false,

    "atac.genome_tsv" : "/path_to_genome_data/hg38/hg38.tsv",

    "atac.paired_end" : true,

    "atac.fastqs_rep1_R1" : [ "rep1_R1_L1.fastq.gz", "rep1_R1_L2.fastq.gz", "rep1_R1_L3.fastq.gz" ],
    "atac.fastqs_rep1_R2" : [ "rep1_R2_L1.fastq.gz", "rep1_R2_L2.fastq.gz", "rep1_R2_L3.fastq.gz" ],
    "atac.fastqs_rep2_R1" : [ "rep2_R1_L1.fastq.gz", "rep2_R1_L2.fastq.gz" ],
    "atac.fastqs_rep2_R2" : [ "rep2_R2_L1.fastq.gz", "rep2_R2_L2.fastq.gz" ],

    "atac.auto_detect_adapter" : false,
    "atac.adapter" : "AATTCCGG",
    "atac.adapters_rep1_R1" : [ "AATTCCGG", "AATTCCGG", "AATTCCGG" ],
    "atac.adapters_rep1_R2" : [ "AATTCCGG", "AATTCCGG" ],
    "atac.adapters_rep2_R1" : [ "AATTCCGG", "AATTCCGG", "AATTCCGG" ],
    "atac.adapters_rep2_R2" : [ "AATTCCGG", "AATTCCGG" ],

    "atac.multimapping" : 4
}

Once you have this completed `.json` file, you could theoretically obtain a pipeline which runs end-to-end quality control and processing of ATAC-seq data, potentially on a computer cluster:

`https://github.com/ENCODE-DCC/atac-seq-pipeline`

We provided this for you here as many of you may end up working with bulk ATAC-seq data in your careers. You can go to this Github page, get the pipeline installed on your local cluster (with the help of systems admins!), and then use any `.json` template you create to execute the pipeline.

For the purposes of more extensive in-class work, we'll turn now to setting up for analyzing scATAC-seq data.

## Set up pipelines for scATAC-seq analysis

Just like for scRNA-seq, many tools have been developed for the analysis of scATAC-seq data. These packages include `Signac`, `ArchR`, `SnapATAC (v1)` in **R** and `SnapATAC (v2)`, `EpiScanpy`, `PeakVI` in **Python**. Since we have used `Seurat` for analyzing scRNA-seq data, we will use `Signac` (developed by the same lab) for the scATAC-seq data analysis. 

Just like `Seurat`, `Signac` has many useful vigenettes as well: https://stuartlab.org/signac/ 

Here, we will be analyzing a single-cell ATAC-seq dataset collected from Human PBMC by 10x Genomics. 

### Loading scATAC-seq data
From the `CellRanger-ATAC` pipeline, four files will be used for constructing the `Signac` object:

- **A count file**. The rows are regions (peaks) and the colums are cells. Each entry [i,j] is the number of reads mapping to region i in cell j. In this assignment, our file is 

`atac_pbmc_500_nextgem_filtered_peak_bc_matrix.h5`


- **A meta data file**, with some overall statistics for each cell. In this assignment, our file is 

`atac_pbmc_500_nextgem_singlecell.csv`


- **A fragment file**, with information on all sequenced fragments (where it maps to the genome, which cell barcode is associated, and how many PCR duplicates were found). In this assignment, our file is 

`atac_pbmc_500_nextgem_fragments_sub.tsv.gz`


- **An index file** connected to the fragment file. This is similar to an index file for a `bam` file, and makes it possible to quickly find fragments for a certain genomic region without having to search the entire file. In this assignment, our file is called 

`atac_pbmc_500_nextgem_fragments_sub.tsv.gz.tbi`


(Note that you won't specify this file below, but it is required. We pre-created this file for you using `tabix` in UNIX.)

### Creating a Seurat object

**Q3.** Using the above files, edit the code below to specify the file names in the places indicated:

In [ ]:
counts <- Read10X_h5(filename = "") ##Specify the count file here

metadata <- read.csv(
  file = "", ##Specify the meta data file here
  header = TRUE,
  row.names = 1
)

chrom_assay <- CreateChromatinAssay(
  counts = counts,
  sep = c(":", "-"),
  fragments = "", ##Specify the fragment file here
  min.cells = 10,
  min.features = 200
)

pbmc <- CreateSeuratObject(
  counts = chrom_assay,
  assay = "peaks",
  meta.data = metadata
)

Next, let's add gene annotations. 

For this, we will take advantage of precomputed annotations which you can search for at `AnnotationHub`.

This will allow downstream functions to pull the gene annotation information directly from the object.

In [ ]:
hub <- AnnotationHub()
query(hub, c("ensdb", "homo sapiens"))

From the above, you can see a list of many annotations you could choose from, which are available for many species. For example, you *could* replace `homo sapiens` in the above with mouse (`mus musculus`) or zebrafish (`danio rerio`) and also obain annotation sets. 

But here, we are working with human data, so let's select a recent annotated genome data base corresponding to the database ID `AH53211`. Let's store that annotation in an object called `ensdb`, and then extract/store our annotations (note: the second cell may take a few seconds to run).

In [ ]:
ensdb <- hub[["AH53211"]]

In [ ]:
# extract gene annotations from EnsDb
annotations <- GetGRangesFromEnsDb(ensdb = ensdb)

# change to UCSC style since the data was mapped to hg38
seqlevels(annotations) <- paste0('chr', seqlevels(annotations))
genome(annotations) <- "hg38"

# add the gene information to our Seurat object
Annotation(pbmc) <- annotations
Annotation(pbmc)

### Computing QC metrics 

In this section, we'll introduce some common quality control metrics. For more information, you can refer to the Signac website (https://stuartlab.org/signac/articles/pbmc_vignette). 

- Nucleosome signal scores were briefly mentioned in the pre-lab notebook, and represent the ratio of mononucleosome regions to nucleosome-free regions. We expect the distribution of different-sized fragments from each region to correspond to the length of DNA wrapped around a single nucleosome; any deviations might indicate a problem with library construction. 

- The enrichment of Tn5 integration events at transcriptional start sites (TSSs) was also mentioned in the pre-lab, and is an important quality control metric to assess the targeting of Tn5 in ATAC-seq experiments. The ENCODE consortium defines a **TSS enrichment score** as the number of Tn5 integration site around the TSS normalized to the number of Tn5 integration sites in flanking regions. See the ENCODE documentation for more information about the TSS enrichment score (https://www.encodeproject.org/data-standards/terms/). We can calculate the TSS enrichment score for each cell using the `TSSEnrichment()` function in Signac.

- We also want to look at the fraction of all fragments that fall within ATAC-seq peaks, as cells with low values (i.e. <15-20%) are either low-quality cells or technical artifacts that should be removed.

- Finally, ENCODE provides a list of blacklist regions (https://github.com/Boyle-Lab/Blacklist) which are often associated with artefactual signal. Cells with a large number of reads mapping to these areas should be removed. 

In [ ]:
### You can ignore any warning messages in this step as long as there is no error

# compute nucleosome signal score per cell
pbmc <- NucleosomeSignal(object = pbmc)

# compute TSS enrichment score per cell
pbmc <- TSSEnrichment(object = pbmc)

# add fraction of reads in peaks
pbmc$pct_reads_in_peaks <- pbmc$peak_region_fragments / pbmc$passed_filters * 100

# add blacklist ratio
pbmc$blacklist_ratio <- FractionCountsInRegion(
   object = pbmc,
   assay = 'peaks',
   regions = blacklist_hg38_unified
)

We can visualize the relationship between variables stored in our object (`pmbc`) using `DensityScatter`. This can be helpful in deciding suitable cutoff values for different QC metrics that we have calculated. For example, let's look at the relationship between read count and TSS enrichment:

In [ ]:
DensityScatter(pbmc, x = 'nCount_peaks', y = 'TSS.enrichment', log_x = TRUE, quantiles = TRUE)

The data here are more *sparse* than what you would see in a typical experiment, as we have created a subset of data that you can work with within the CoCalc environment. However, what you can see is that there is a central density of data, with some outliers (high and low peak counts). 

Next, it is also helpful to look at the fragment length distribution, as we expect nucleosome positioning periodicity in the data. We can look at this using the `nucleosome_signal` we computed earlier; let's look at characteristics for scores less than 4 and greater than 4:

In [ ]:
pbmc$nucleosome_group <- ifelse(pbmc$nucleosome_signal > 4, 'NS > 4', 'NS < 4')
FragmentHistogram(object = pbmc, group.by = 'nucleosome_group')

The plot on the left shows that for NS < 4, we can see the periodicity we expect from a successful ATAC-Seq experiment. 

**Q4.** Given the following approximate fragment sizes for different regions: 
- Nucleosome-free regions (NFR): ~50–100 bp
- Mononucleosome fragments: ~180 bp (approximately the length of DNA wrapped around a single nucleosome)
- Dinucleosome fragments: ~360 bp
- Trinucleosome fragments: ~540 bp

Do you see a bias towards any particular type of fragment/region in the histogram on the right (NS > 4)?

Now, let's plot the distribution of each QC metric separately via a violin plot:

In [ ]:
VlnPlot(
  object = pbmc,
  features = c('nCount_peaks', 'TSS.enrichment', 'blacklist_ratio', 'nucleosome_signal', 'pct_reads_in_peaks'),
  pt.size = 0.1,
  ncol = 3
)

**Q5.** Do the results reflect what you expect to see? For each QC metric, briefly explain the distribution if values you expect to see (high, low, within a certain range) and whether or not the actual plotted values match. 

### Filtering based on QC metrics

Here, we will set some initial QC filters, to focus on a subset of peaks with the following properties:

- read count in peaks greater than 9,000 (i.e., insist on a minimum read depth in a peak)
- read count in peaks less than 100,000 (i.e., if a peak has too much depth, exclude)
- minimum fraction of read coverage in peaks at least 40%
- blacklist ratio less than 1%
- nucleosome signal less than 4
- TSS enrichemnt greater than 4

In [ ]:
pbmc <- subset(
  x = pbmc,
  subset = nCount_peaks > 9000 &
    nCount_peaks < 100000 &
    pct_reads_in_peaks > 40 &
    blacklist_ratio < 0.01 &
    nucleosome_signal < 4 &
    TSS.enrichment > 4
)
pbmc

### Normalization and linear dimensionality reduction

Next, as described in the pre-lab, we will normalize the subset of peaks we selected above using Term Frequency-Inverse Document Frequency (TF-IDF) scores, and then perform dimensionality reduction for interpretive purposes using singular value decomposition (SVD). Together, this method is called **Latent Semantic Indexing** (LSI). 

In [ ]:
pbmc <- RunTFIDF(pbmc)
pbmc <- FindTopFeatures(pbmc, min.cutoff = 'q0')
pbmc <- RunSVD(pbmc)

The first LSI component often captures sequencing depth (technical variation) rather than biological variation. If this is the case, the component should be removed from downstream analysis. We can assess the correlation between each LSI component and sequencing depth using the DepthCor() function:

In [ ]:
DepthCor(pbmc)

**Q6.** What is the correlation of the first LSI component with the total number of counts per cell? Do you think it should be removed from downstream analysis?

### Non-linear dimension reduction and clustering
Now that the cells are embedded in a low-dimensional space, we can use methods commonly applied for the analysis of scRNA-seq data to perform graph-based clustering and non-linear dimension reduction for visualization. The functions `RunUMAP()`, `FindNeighbors()`, and `FindClusters()` all come from the Seurat package.

In [ ]:
pbmc <- RunUMAP(object = pbmc, reduction = 'lsi', dims = 2:30)
pbmc <- FindNeighbors(object = pbmc, reduction = 'lsi', dims = 2:30)
pbmc <- FindClusters(object = pbmc, verbose = FALSE, algorithm = 3)
DimPlot(object = pbmc, label = TRUE) + NoLegend()

**Note**: In this notebook, we will skip the integration with scRNA-seq part. Based on various benchmark efforts, the integration between unmatched scRNA-seq and scATAC-seq can be very challenging. We recommend running this type of analysis with caution. 

### Finding differentially accessible peaks between clusters
To find differentially accessible regions between clusters of cells, we can perform a **differential accessibility** (DA) test. A simple approach is to perform a Wilcoxon rank sum test, and the `presto` package has implemented an extremely fast Wilcoxon test that can be run on a Seurat object.

In [ ]:
DefaultAssay(pbmc) <- 'peaks'

da_peaks <- FindMarkers(
  object = pbmc,
  ident.1 = c("0"), 
  ident.2 = c("1"), 
  test.use = 'wilcox',
  min.pct = 0.1
)

head(da_peaks)

Run the code below to generate a violin plot. 

In [ ]:
plot1 <- VlnPlot(
  object = pbmc,
  features = rownames(da_peaks)[1],
  pt.size = 0.1,
  idents = c("0","1")
)
plot1

**Q7.** Describe what the violin plot above is showing. Does the visualization make sense? 

(Hint: What are the x-axis and y-axis? Where does the title come from?)

Finally, let's look at the annotations for some of these significant peaks. We can look at where cluster 0 is open, relative to cluster 1, and vice-versa (where cluster 1 is open, realtive to cluster 0).

Let us also filter the results by significance of association as well as the log fold change:

In [ ]:
open_c0 <- rownames(da_peaks[da_peaks$avg_log2FC > 3 & da_peaks$p_val_adj < 1e-5, ])
open_c1 <- rownames(da_peaks[da_peaks$avg_log2FC < -3 & da_peaks$p_val_adj < 1e-5, ])

closest_genes_c0 <- ClosestFeature(pbmc, regions = open_c0)
closest_genes_c1 <- ClosestFeature(pbmc, regions = open_c1)

head(closest_genes_c0)
head(closest_genes_c1)

### Visualizing gene accessibility across clusters
Often times, we want to display the accessibility landscape across clusters (or cell types). There are several different plot types for this beyond the violin plot. 

1. **Feature plot**: This plot will show accessibility as colored dots in a low dimensional embedding (such as tSNE or UMAP). Each dot represent a cell, and the color represents the accessibility level of the cell. This is the plot that we'll generate below.
2. **Peak pileup plot**: This plot looks like the ones you see in the UCSC Genome Browser or Integrative Genomics Viewer (IGV). This plot typically shows more detail of the actual insertion location and the shape of the peak. This also enables you to add other genomic tracks to it. You can find some example figures in https://www.nature.com/articles/s41467-021-22266-1/figures/6.
3. **Interactive plots**: You can upload your tracks to IGV to obtain an interactive interface where you can move along genomic tracks. For example: https://susztaklab.com/developing_adult_kidney/igv/. (Note, there is also a local version of IGV!)

Signac has provided functions for generating these plots. Please refer to this website for more details: https://stuartlab.org/signac/articles/visualization 

In [ ]:
# an example feature plot
plot2 <- FeaturePlot(
  object = pbmc,
  features = rownames(da_peaks)[1],
  pt.size = 0.1
)
plot2

**Q8.** Referencing the cluster labels from the UMAP visualization we generated earlier, describe what you see in this feature plot. Are the results consistent with the violin plot from Q7? 

### Gene Ontology Enrichment Analysis
Just like in the proteomics module, we can also perform enrichment analyses for our peaks using the gene TSS that is the nearest to our peaks. We have done this below for c0 and c1 clusters -- but you could do comparisons with different clusters by editing the code above and below. 

These two cells may take a while to run. 

In [ ]:
cd0_ego <- enrichGO(gene = closest_genes_c0$gene_id,
                keyType = "ENSEMBL",
                OrgDb = org.Hs.eg.db,
                ont = "BP",
                pAdjustMethod = "BH",
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.05,
                readable = TRUE)

barplot(cd0_ego,showCategory = 20)

In [ ]:
cd1_ego <- enrichGO(gene = closest_genes_c1$gene_id,
                keyType = "ENSEMBL",
                OrgDb = org.Hs.eg.db,
                ont = "BP",
                pAdjustMethod = "BH",
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.05,
                readable = TRUE)

barplot(cd1_ego,showCategory = 20)